# Analisi Esplorativa dei Dati (EDA)

**Obiettivo**: caratterizzare le proprietà statistiche del dataset che guidano direttamente la scelta dei metodi di raccomandazione (Baseline → Content-Based → Collaborative Filtering → Hybrid).

Analizziamo:
1) dimensione del dataset e copertura temporale,
2) struttura dei candidate set e squilibrio dei click,
3) cronologia utente (cold-start),
4) ricchezza dei contenuti testuali (testo + entità),
5) vita degli articoli / decadimento temporale,
6) sparsità delle interazioni (limiti del Collaborative Filtering puro).

Carichiamo i due file di `MINDsmall_train`:
- `news.tsv`: una riga per articolo, con categoria, titolo/abstract ed entità annotate.
- `behaviors.tsv`: una riga per impression, con timestamp, cronologia di click dell'utente e articoli candidati con relativa etichetta (`-1` cliccato, `-0` non cliccato).

Questa struttura rispecchia la formulazione standard del benchmark MIND: ogni impression è un task di ranking sui candidati mostrati.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
# Carica i dati di MINDsmall_train (news.tsv, behaviors.tsv) con schema esplicito.
# Usiamo percorsi robusti: dataset/MINDsmall_train/... oppure file locali nella cartella del notebook.
# Output: news_df (metadata e contenuto degli articoli), behav_df (impression + cronologia).
import seaborn as sns  # necessario per countplot/histplot più avanti
import json
from pathlib import Path
from collections import Counter

# Nomi delle colonne (schema MIND)
news_cols = ['news_id', 'category', 'subcategory', 'title', 'abstract', 'url', 'title_entities', 'abstract_entities']
behav_cols = ['imp_id', 'user_id', 'time', 'history', 'impressions']

# ---- PERCORSI (robusti) ----
DATA_DIR = Path("dataset/MINDsmall_train")  # se la cartella è presente
NEWS_PATH = DATA_DIR / "news.tsv"
BEH_PATH  = DATA_DIR / "behaviors.tsv"

# Fallback: usa file locali accanto al notebook se dataset/ non è disponibile
if not NEWS_PATH.exists():
    NEWS_PATH = Path("news.tsv")
if not BEH_PATH.exists():
    BEH_PATH = Path("behaviors.tsv")

news_df  = pd.read_csv(NEWS_PATH, sep="\t", header=None, names=news_cols)
behav_df = pd.read_csv(BEH_PATH,  sep="\t", header=None, names=behav_cols)

print("Righe news:", len(news_df))
print("Righe behaviors:", len(behav_df))
print("Utenti unici:", behav_df["user_id"].nunique())
print("News uniche in news.tsv:", news_df["news_id"].nunique())

In [ ]:
# Preprocessing di behaviors.tsv:
# 1) parsing dei timestamp,
# 2) tokenizzazione della cronologia utente e delle liste di candidati per impression,
# 3) calcolo delle statistiche per impression: history_length, n_candidates, n_clicks e CTR globale.
# Queste feature vengono riutilizzate in tutte le sezioni successive.

# ---- Datetime ----
behav_df["time"] = pd.to_datetime(behav_df["time"], errors="coerce")

print("Intervallo temporale:", behav_df["time"].min(), "->", behav_df["time"].max())

# ---- Tokenizza cronologia e impression in liste ----
behav_df["history_list"] = behav_df["history"].fillna("").astype(str).str.split()
behav_df["history_length"] = behav_df["history_list"].str.len()

behav_df["impressions_list"] = behav_df["impressions"].fillna("").astype(str).str.split()
behav_df["n_candidates"] = behav_df["impressions_list"].str.len()

# click per impression = quanti elementi terminano con "-1"
behav_df["n_clicks"] = behav_df["impressions_list"].apply(lambda lst: sum(x.endswith("-1") for x in lst))

total_candidates = behav_df["n_candidates"].sum()
total_clicks = behav_df["n_clicks"].sum()
ctr_global = total_clicks / total_candidates if total_candidates > 0 else 0.0

print("Candidati medi/impression:", behav_df["n_candidates"].mean())
print("Candidati mediani/impression:", behav_df["n_candidates"].median())
print("Click medi/impression:", behav_df["n_clicks"].mean())
print("Click mediani/impression:", behav_df["n_clicks"].median())
print("CTR globale (click/mostrati):", ctr_global)

### Statistiche principali

In [ ]:
# --- Tabella di riepilogo (panoramica del dataset) ---
n_news = news_df["news_id"].nunique()
n_users = behav_df["user_id"].nunique()
n_impressions = len(behav_df)
time_min = behav_df["time"].min()
time_max = behav_df["time"].max()
time_span_days = (time_max - time_min).total_seconds() / (3600 * 24)
total_candidates = behav_df["n_candidates"].sum()
total_clicks = behav_df["n_clicks"].sum()
ctr_global = total_clicks / total_candidates if total_candidates else float("nan")
# Coppie distinte (utente, news) cliccate => sparsità
# Costruisce un `inter_df` minimale se non esiste ancora (l'ordine di preprocessing può variare)
if 'inter_df' not in globals():
    expl = behav_df[["imp_id", "user_id", "time", "impressions_list"]].explode("impressions_list")
    expl = expl[expl["impressions_list"].notna() & (expl["impressions_list"] != "")].copy()
    tmp = expl["impressions_list"].str.rsplit("-", n=1, expand=True)
    expl["news_id"] = tmp[0]
    expl["clicked"] = tmp[1].astype(int)
    inter_df = expl[["imp_id", "user_id", "time", "news_id", "clicked"]].copy()
click_pairs = inter_df[inter_df["clicked"] == 1][["user_id", "news_id"]].drop_duplicates()
density = len(click_pairs) / (n_users * n_news)
# Vita degli articoli (richiede prima/ultima esposizione; se non calcolata, la calcola qui)
news_seen = inter_df.groupby("news_id")["time"].agg(["min", "max"])
lifetime_hours = (news_seen["max"] - news_seen["min"]).dt.total_seconds() / 3600

summary = pd.DataFrame({
    "metric": [
        "#news", "#users", "#impressions",
        "time_range_start", "time_range_end", "time_span_days",
        "total_candidates", "total_clicks", "global_CTR",
        "distinct(user,news)_click_pairs", "user_item_density",
        "news_lifetime_median_h", "news_lifetime_mean_h", "share_lifetime<=48h_%"
    ],
    "value": [
        n_news, n_users, n_impressions,
        time_min, time_max, time_span_days,
        total_candidates, total_clicks, ctr_global,
        len(click_pairs), density,
        lifetime_hours.median(), lifetime_hours.mean(), 100 * (lifetime_hours <= 48).mean()
    ]
})

summary.loc[summary["metric"] == "global_CTR", "value"] = 100 * summary.loc[summary["metric"] == "global_CTR", "value"]
summary.loc[summary["metric"] == "global_CTR", "metric"] = "global_CTR_%"
display(summary)

### Analisi del contenuto: categorie delle news

Prima di analizzare i click, osserviamo la distribuzione degli articoli per categoria.
Lo squilibrio tra categorie influisce su:
- le baseline di popolarità (alcune categorie dominano l'esposizione),
- i profili utente (interessi verticali/di nicchia),
- la stabilità della valutazione (le categorie a basso volume sono più rumorose).

In [ ]:
# Distribuzione delle categorie (news.tsv).
# Obiettivo: quantificare lo squilibrio tra categorie (head vs tail), che influenza le baseline di popolarità.

# Analisi della distribuzione delle categorie
plt.figure(figsize=(12, 6))
sns.countplot(data=news_df, y='category', order=news_df['category'].value_counts().index)
plt.title('Distribuzione delle categorie di news')
plt.xlabel('Numero di articoli')
plt.show()

In [ ]:
# Statistiche testuali (news.tsv):
# - conteggio parole per titolo e abstract,
# - frazione di abstract mancanti,
# - istogrammi che mostrano quanto sia breve il contenuto (in particolare i titoli).
# Questo motiva l'adozione di modelli content-based capaci di lavorare con testo sparso.

# ---- Statistiche sulla lunghezza del testo ----
news_df["title_len_words"] = news_df["title"].fillna("").astype(str).str.split().str.len()
news_df["abstract_len_words"] = news_df["abstract"].fillna("").astype(str).str.split().str.len()

news_df["abstract_missing"] = news_df["abstract"].isna() | (news_df["abstract"].astype(str).str.strip() == "")
print("Abstract mancanti (%):", 100 * news_df["abstract_missing"].mean())

print("Parole nel titolo: media", news_df["title_len_words"].mean(), "mediana", news_df["title_len_words"].median())
print("Parole nell'abstract: media", news_df["abstract_len_words"].mean(), "mediana", news_df["abstract_len_words"].median())

plt.figure(figsize=(8,4))
plt.hist(news_df["title_len_words"], bins=60)
plt.title("Lunghezza del titolo (parole)")
plt.xlabel("# parole")
plt.ylabel("conteggio")
plt.show()

plt.figure(figsize=(8,4))
plt.hist(news_df["abstract_len_words"], bins=80)
plt.title("Lunghezza dell'abstract (parole)")
plt.xlabel("# parole")
plt.ylabel("conteggio")
plt.show()

In [ ]:
# Statistiche sulle entità (news.tsv):
# Esegue il parsing delle liste JSON in title_entities / abstract_entities e conta le entità.
# Riportiamo il conteggio medio e la percentuale di articoli senza alcuna entità.
# Questo indica quanto spesso i segnali basati su entità siano disponibili per la raccomandazione.

def safe_entity_count(s):
    if pd.isna(s):
        return 0
    s = str(s).strip()
    if s == "" or s == "nan":
        return 0
    try:
        obj = json.loads(s)
        return len(obj) if isinstance(obj, list) else 0
    except Exception:
        return 0

news_df["n_title_entities"] = news_df["title_entities"].apply(safe_entity_count)
news_df["n_abs_entities"] = news_df["abstract_entities"].apply(safe_entity_count)

print("Entità nel titolo: media", news_df["n_title_entities"].mean(), "zero-%", 100*(news_df["n_title_entities"]==0).mean())
print("Entità nell'abstract: media", news_df["n_abs_entities"].mean(), "zero-%", 100*(news_df["n_abs_entities"]==0).mean())

plt.figure(figsize=(8,4))
plt.hist(news_df["n_title_entities"], bins=30)
plt.title("Numero di entità nel titolo")
plt.xlabel("# entità")
plt.ylabel("conteggio")
plt.show()

plt.figure(figsize=(8,4))
plt.hist(news_df["n_abs_entities"], bins=30)
plt.title("Numero di entità nell'abstract")
plt.xlabel("# entità")
plt.ylabel("conteggio")
plt.show()

### Analisi della cronologia utente

Misuriamo quanta informazione storica è disponibile per ciascun utente al momento di ogni impression.
Questo determina:
- quanto è severo il problema del cold-start,
- se i modelli centrati sull'utente (es. attention / pesatura per recency) sono realisticamente applicabili.

In [ ]:
# Distribuzione della lunghezza della cronologia utente (behaviors.tsv).
# Qui ricalcoliamo history_length direttamente dalla stringa 'history' grezza, per rendere questa sezione autonoma.
# Output: istogramma + tabella della quota di impression per fascia.

# Distribuzione della lunghezza della cronologia utente
behav_df['history_length'] = behav_df['history'].apply(lambda x: len(str(x).split()) if pd.notna(x) else 0)

plt.figure(figsize=(10, 5))
sns.histplot(behav_df['history_length'], bins=50, kde=True)
plt.title('Distribuzione della lunghezza della cronologia utente')
plt.xlabel('Numero di news cliccate in precedenza')
plt.show()

# ---- Fasce di cold-start (quota di impression per lunghezza di cronologia) ----
bins = [-1, 0, 5, 10, 20, 50, 999999]
labels = ["0", "1-5", "6-10", "11-20", "21-50", "50+"]

behav_df["history_bucket"] = pd.cut(behav_df["history_length"], bins=bins, labels=labels)

bucket_stats = behav_df["history_bucket"].value_counts(normalize=True).sort_index().to_frame("share_impressions")
bucket_stats["share_impressions"] = bucket_stats["share_impressions"] * 100
display(bucket_stats)

### Elaborazione delle impression

Ogni impression fornisce un insieme di candidati nella forma: `Nxxxx-1 Nyyyy-0 ...`.
Trasformiamo le impression in una tabella a livello di interazione (`inter_df`), con una riga per ciascun articolo mostrato: (imp_id, user_id, time, news_id, clicked).
Questo formato viene riutilizzato per le analisi di popolarità, CTR, vita degli articoli e dinamiche temporali.

In [ ]:
# Costruisce inter_df espandendo le liste di candidati delle impression.
# inter_df è la tabella centrale degli "articoli mostrati" (una riga per candidato per impression).
# Vengono inoltre visualizzati:
# - la distribuzione della dimensione del candidate set,
# - la distribuzione dei click per impression.
# Questi grafici descrivono lo squilibrio delle etichette e la difficoltà del task di ranking.

# ---- Espande le impression: una riga per (imp_id, user_id, time, news_id, clicked) ----
expl = behav_df[["imp_id", "user_id", "time", "impressions_list"]].explode("impressions_list")
expl = expl[expl["impressions_list"].notna() & (expl["impressions_list"] != "")].copy()

# separa "Nxxxxx-0/1"
tmp = expl["impressions_list"].str.rsplit("-", n=1, expand=True)
expl["news_id"] = tmp[0]
expl["clicked"] = tmp[1].astype(int)

inter_df = expl[["imp_id", "user_id", "time", "news_id", "clicked"]].copy()
print("Righe di interazione (articoli mostrati):", len(inter_df))

# Distribuzione del numero di candidati per impression
plt.figure(figsize=(8,4))
plt.hist(behav_df["n_candidates"], bins=60)
plt.title("Candidati per impression")
plt.xlabel("# candidati")
plt.ylabel("conteggio")
plt.show()

# Distribuzione del numero di click per impression
plt.figure(figsize=(8,4))
plt.hist(behav_df["n_clicks"], bins=30)
plt.title("Click per impression")
plt.xlabel("# click")
plt.ylabel("conteggio")
plt.show()

### Riepilogo della distribuzione candidati/click

In [ ]:
# --- Riepilogo della distribuzione candidati/click (quantili ed estremi) ---
dist_tbl = pd.DataFrame({
    "stat": ["min", "p25", "median", "p75", "max", "mean"],
    "n_candidates": [
        behav_df["n_candidates"].min(),
        behav_df["n_candidates"].quantile(0.25),
        behav_df["n_candidates"].median(),
        behav_df["n_candidates"].quantile(0.75),
        behav_df["n_candidates"].max(),
        behav_df["n_candidates"].mean()
    ],
    "n_clicks": [
        behav_df["n_clicks"].min(),
        behav_df["n_clicks"].quantile(0.25),
        behav_df["n_clicks"].median(),
        behav_df["n_clicks"].quantile(0.75),
        behav_df["n_clicks"].max(),
        behav_df["n_clicks"].mean()
    ],
})

display(dist_tbl)

**Nota (struttura delle etichette).** In questo split, ogni impression contiene almeno un elemento cliccato (etichetta positiva).
Di conseguenza, ogni istanza di ranking ha sempre almeno un positivo, e lo squilibrio principale deriva dal gran numero di negativi per impression (CTR basso).

In [ ]:
# Verifica se alcune impression hanno zero click
zero_click_share = (behav_df["n_clicks"] == 0).mean() * 100
print("Quota di impression con 0 click (%):", zero_click_share)

### Individuazione delle news più popolari per la baseline

Calcoliamo un segnale semplice ma efficace: il conteggio globale dei click per articolo.
Questo supporta:
- una baseline di raccomandazione Most-Popular,
- un'analisi head/tail della distribuzione (comportamento a legge di potenza).

In [ ]:
# Baseline di popolarità: conta i click per news_id (clicked==1) e mostra i primi 15 articoli con titolo/categoria.
# Fornisce un riferimento per il recommender "Most Popular".

# ---- Baseline di popolarità: conteggio dei click per articolo ----
click_counts = inter_df[inter_df["clicked"] == 1]["news_id"].value_counts().rename("click_count").reset_index()
click_counts = click_counts.rename(columns={"index": "news_id"})

top_15 = click_counts.head(15).merge(news_df[["news_id", "title", "category"]], on="news_id", how="left")
display(top_15)

### Analisi del CTR per categoria

Il CTR è calcolato come click / impression all'interno di ciascuna categoria.
Ci concentriamo sulle categorie a volume più alto, per evitare stime instabili dovute a categorie rare.
Questo evidenzia quali categorie attraggono un numero di click sproporzionatamente alto rispetto alla loro esposizione.

In [ ]:
# Calcola il CTR per categoria unendo inter_df con news_df.
# Output:
# - una tabella con total_impressions, total_clicks, CTR%
# - un grafico a barre per le 10 categorie con maggior volume di esposizione.

# ---- CTR per categoria (categorie principali per volume) ----
inter_cat = inter_df.merge(news_df[["news_id", "category"]], on="news_id", how="left")

cat_stats = inter_cat.groupby("category").agg(
    total_impressions=("clicked", "count"),
    total_clicks=("clicked", "sum")
)
cat_stats["CTR_%"] = 100 * cat_stats["total_clicks"] / cat_stats["total_impressions"]
top_cat_stats = cat_stats.sort_values("total_impressions", ascending=False).head(10)

display(top_cat_stats)

plt.figure(figsize=(10,5))
plt.barh(top_cat_stats.index.astype(str), top_cat_stats["CTR_%"])
plt.title("CTR per categoria (prime 10 per volume)")
plt.xlabel("CTR (%)")
plt.gca().invert_yaxis()
plt.show()

### Analisi temporale

La raccomandazione di news dipende fortemente dal tempo.
Analizziamo:
- il volume di impression nel tempo (su base oraria),
- la vita degli articoli (prima/ultima esposizione),
per quantificare il decadimento temporale e motivare baseline/modelli sensibili alla recency.

In [ ]:
# Attività delle impression nel tempo (aggregazione oraria).
# Obiettivo: verificare la copertura temporale e individuare i picchi che influenzano popolarità/CTR.

# Converte time in oggetti datetime
behav_df['time'] = pd.to_datetime(behav_df['time'])
print(f"I dati coprono il periodo da {behav_df['time'].min()} a {behav_df['time'].max()}")

# Grafico dell'attività nel tempo
behav_df.set_index('time').resample('h').size().plot(title='Interazioni utente per ora')
plt.ylabel('Numero di impression')
plt.show()

In [ ]:
# Stima della vita degli articoli:
# per ogni news_id, calcola first_seen e last_seen tra le esposizioni come candidato.
# lifetime_hours = (last_seen - first_seen).
# Riportiamo vita mediana/media e la quota di articoli con vita <= 48h, per quantificare il decadimento rapido.

# ---- Vita degli articoli: prima/ultima volta che una news compare tra i candidati ----
news_seen = inter_df.groupby("news_id")["time"].agg(["min", "max"]).rename(columns={"min":"first_seen", "max":"last_seen"})
news_seen["lifetime_hours"] = (news_seen["last_seen"] - news_seen["first_seen"]).dt.total_seconds() / 3600.0

print("Vita in ore: mediana", news_seen["lifetime_hours"].median(), "media", news_seen["lifetime_hours"].mean())
print("Quota <= 48h (%):", 100 * (news_seen["lifetime_hours"] <= 48).mean())

plt.figure(figsize=(8,4))
plt.hist(news_seen["lifetime_hours"].dropna(), bins=100)
plt.title("Vita delle news (ore), basata sulle esposizioni come candidato")
plt.xlabel("ore")
plt.ylabel("conteggio")
plt.show()

In [ ]:
# Sparsità delle interazioni (solo click):
# calcola le coppie distinte (user_id, news_id) cliccate e stima la densità della matrice:
# densità = coppie / (num_utenti * num_news).
# Questo quantifica quanto sia difficile il Collaborative Filtering puro nella raccomandazione di news.

# ---- Sparsità delle interazioni utente-item (solo click) ----
click_pairs = inter_df[inter_df["clicked"] == 1][["user_id", "news_id"]].drop_duplicates()
n_click_pairs = len(click_pairs)

n_users = behav_df["user_id"].nunique()
n_news = news_df["news_id"].nunique()

density = n_click_pairs / (n_users * n_news)
print("Coppie distinte (utente, news) cliccate:", n_click_pairs)
print("Densità utente-item:", density)

### Heatmap dei click per categoria e ora del giorno

Visualizziamo in quali fasce orarie vengono cliccate le diverse categorie.
Ci limitiamo agli articoli effettivamente cliccati, per studiare i pattern di engagement e non la semplice esposizione.

In [ ]:
# Costruisce un log di eventi contenente solo i click:
# per ogni impression, estrae i news_id cliccati e li memorizza come (time, news_id).
# Poi unisce con la categoria, estrae l'ora e crea una pivot table (ora x categoria) per la heatmap.
# Obiettivo: individuare i pattern orari di engagement degli utenti per categoria.

# Dobbiamo collegare i click alle loro categorie e timestamp
# Espandiamo le impression per associare ogni click al proprio orario
click_data = []
for _, row in behav_df.iterrows():
    if pd.isna(row['impressions']): continue

    # Estrae solo le news cliccate (etichetta -1)
    clicked_ids = [imp.split('-')[0] for imp in row['impressions'].split() if imp.endswith('-1')]
    for n_id in clicked_ids:
        click_data.append({'time': row['time'], 'news_id': n_id})

click_events_df = pd.DataFrame(click_data)

# Unisce con news_df per ottenere le categorie
click_events_df = click_events_df.merge(news_df[['news_id', 'category']], on='news_id')

# Estrae l'ora del giorno
click_events_df['hour'] = click_events_df['time'].dt.hour

# Crea una pivot table per la heatmap
# Ci concentriamo sulle prime 10 categorie per mantenere la visualizzazione leggibile
top_categories = click_events_df['category'].value_counts().nlargest(10).index
pivot_df = click_events_df[click_events_df['category'].isin(top_categories)]
pivot_table = pivot_df.pivot_table(index='category', columns='hour', values='news_id', aggfunc='count').fillna(0)

# Visualizzazione
plt.figure(figsize=(15, 8))
sns.heatmap(pivot_table, cmap='YlGnBu', annot=False, cbar_kws={'label': 'Numero di click'})
plt.title('Heatmap: click sulle news per categoria e ora del giorno', fontsize=15)
plt.xlabel('Ora del giorno (formato 24h)', fontsize=12)
plt.ylabel('Categoria news', fontsize=12)
plt.show()

## Conclusioni

L'esplorazione del dataset MIND permette di individuare alcune caratteristiche strutturali che motivano direttamente le scelte metodologiche del progetto e sono coerenti con quanto riportato dalla documentazione ufficiale del dataset.

**1) Forte squilibrio nei task di ranking.**
Ogni impression contiene molti candidati ma pochi click (CTR globale basso). Questo comporta uno squilibrio di classe marcato e motiva l'uso di strategie di campionamento/pesatura durante il training.

**2) Cold-start non trascurabile / utenti con cronologia breve.**
Una quota significativa di impression coinvolge utenti con cronologia di click breve o assente. Per questi casi, i segnali di popolarità/recency e i segnali content-based sono determinanti.

**3) Vita breve degli articoli e forte decadimento temporale.**
La maggior parte degli articoli si esaurisce entro una finestra temporale ristretta (vita mediana di poche ore, ~90% degli articoli entro 48 ore). Questo giustifica l'adozione di baseline sensibili alla recency e di una modellazione utente time-aware.

**4) Matrice di interazione utente-item estremamente sparsa.**
La densità delle coppie (utente, articolo) osservate è molto bassa, il che limita fortemente l'efficacia del Collaborative Filtering puro. I modelli che sfruttano il contenuto (titolo/abstract/entità) sono strutturalmente più adatti a questo dominio.

**5) Effetti di categoria ("la legge di potenza" della popolarità delle news).**
La distribuzione dei click è fortemente asimmetrica: una piccola frazione di articoli concentra la maggior parte delle interazioni. Categoria e CTR per categoria mostrano che esposizione e coinvolgimento non sono uniformi, un aspetto da considerare nella progettazione dei modelli ibridi.

**6) Pattern temporali circadiani.**
Dalla heatmap emergono picchi di lettura ricorrenti, tra cui un incremento a pranzo (12:00-13:00) e un picco serale (18:00-21:00), spesso legati a categorie diverse (es. Finance al mattino, Lifestyle/TV alla sera).

**7) Il contenuto conta più del semplice comportamento, ma non è l'unico fattore.**
Il testo delle news (titolo, abstract, entità) è ricco ma spesso breve e incompleto; allo stesso tempo, la posizione di un articolo nella lista dei candidati e i trend del momento risultano predittori del click spesso più forti della sola similarità testuale con la cronologia dell'utente (position bias).

**In sintesi**, questi risultati giustificano la progressione metodologica adottata nel progetto:
**Baseline (popolarità/recency) → Content-Based (testo) → Collaborative Filtering (dove applicabile) → Hybrid (migliore copertura e robustezza)**, con particolare attenzione alla gestione del cold-start e alla sensibilità temporale in ciascun modello.